In [0]:
# Import secrets
user = dbutils.secrets.get(scope="snowflake-scope", key="snowflake_user")
password = dbutils.secrets.get(scope="snowflake-scope", key="snowflake_password")
account = dbutils.secrets.get(scope="snowflake-scope", key="snowflake_account")
warehouse = dbutils.secrets.get(scope="snowflake-scope", key="snowflake_warehouse")
database = dbutils.secrets.get(scope="snowflake-scope", key="snowflake_database")
schema = "ANALYTICS"  # Replace with your Snowflake schema name

import snowflake.connector

conn = snowflake.connector.connect(
    user=user,
    password=password,
    account=account,
    warehouse=warehouse,
    database=database,
    schema=schema

)

cursor = conn.cursor()



In [0]:
snowflake_tables = [

    {
        "table": "GOLD_DAILY_METRICS",
        "stage": "daily_data/gold_daily_metrics"
    },

    {
        "table": "GOLD_HOURLY_METRICS",
        "stage": "hourly_data/gold_hourly_metrics"
    },

    {
        "table": "GOLD_PAYMENT_METRICS",
        "stage": "payment_data/gold_payment_metrics"
    },

    {
        "table": "GOLD_VENDOR_METRICS",
        "stage": "Vendor_analysis/gold_vendor_metrics"
    },

    {
        "table": "GOLD_DASHBOARD_METRICS",
        "stage": "dashboard/gold_dashboard_metrics_aws"
    },

    {
        "table": "PIPELINE_MONITORING",
        "stage": "monitoring/pipeline_monitoring"
    }

]

####SQL Generator

In [0]:
def build_refresh_sql(
    table_name,
    stage_path
):

    return f"""
TRUNCATE TABLE {table_name};

COPY INTO {table_name}
FROM @analytics_stage/{stage_path}/
FILE_FORMAT=(FORMAT_NAME='parquet_format')
MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE
PATTERN='.*\\\\.parquet';
"""

#### Refreshing table names

In [0]:
for table in snowflake_tables:

    table_name = table["table"]

    stage = table["stage"]

    sql = build_refresh_sql(
        table_name,
        stage
    )

    print(f"Refreshing {table_name}")

    try:

        statements = [
            s.strip()
            for s in sql.split(";")
            if s.strip()
        ]

        for statement in statements:

            cursor.execute(statement)

        print("✓ Success")

    except Exception as e:

        print(e)

Refreshing GOLD_DAILY_METRICS
✓ Success
Refreshing GOLD_HOURLY_METRICS
✓ Success
Refreshing GOLD_PAYMENT_METRICS
✓ Success
Refreshing GOLD_VENDOR_METRICS
✓ Success
Refreshing GOLD_DASHBOARD_METRICS
✓ Success
Refreshing PIPELINE_MONITORING
✓ Success


#### Validation

In [0]:
for table in snowflake_tables:

    cursor.execute(

        f"""
        SELECT COUNT(*)
        FROM {table['table']}
        """

    )

    count = cursor.fetchone()[0]

    print(
        f"{table['table']:<35}{count:,}"
    )

GOLD_DAILY_METRICS                 309
GOLD_HOURLY_METRICS                24
GOLD_PAYMENT_METRICS               5
GOLD_VENDOR_METRICS                3
GOLD_DASHBOARD_METRICS             309
PIPELINE_MONITORING                72
